
# USPS digits benchmark


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.usps_benchmarks import run_benchmark
from src.utils import plotting_params  

# automatic reload
%load_ext autoreload
%autoreload 2


In [ ]:
out_dir = "../../writeup/Kernel_PCA_Preimage_IEEE_TPAMI/"

## Comparison metrics

In [ ]:
kernels = ['rbf', 'poly', 'sigmoid', 'cosine', 'linear']
noise_levels = [0.0, 0.1, 0.25, 0.5]
n_components = [50, 100]
methods = ['bakir', 'honeine']

sigma = 40
gamma = 1 / (2 * sigma**2)
degree = 3

In [ ]:
for i in range(len(kernels)):
    for j in range(len(noise_levels)):
        for k in range(len(n_components)):
            print(f"Kernel: {kernels[i]} | noise scale: {noise_levels[j]} | number of components: {n_components[k]}")
            _, metrics = run_benchmark(kernel=kernels[i], n_components=int(n_components[k]), noise_scale=noise_levels[j],
                                       gamma=gamma, degree=degree, noise_train=True, methods=methods, n_alpha=100, n_val=100)
            metrics['kernel'] = kernels[i]
            metrics['noise_std'] = noise_levels[j]
            metrics['components'] = n_components[k]

            if i==j==k==0:
                df = pd.DataFrame(metrics)
            else:
                df = pd.concat([df, pd.DataFrame(metrics)], axis=0)
    print('-'*100)

indices = ['MSE', 'SSIM', 'PSNR'] * (len(kernels) * len(noise_levels) * len(n_components))
df.index = indices


### Generate a Latex table 

In [ ]:
# ── display names ─────────────────────────────────────────────────────
method_cols = ["bakir", "honeine", "ours"]
method_names = {
    "bakir": r"BWS",
    "honeine": r"HR",
    "ours": r"Ours",
}

kernel_names = {
    "rbf": "RBF",
    "poly": "Poly",
    "sigmoid": "Sigmoid",
    "cosine": "Cosine",
    "linear": "Linear",
}
kernel_order = ["rbf", "poly", "sigmoid", "cosine", "linear"]

noise_levels = [0.0, 0.1, 0.25, 0.5]
comp_levels = [50, 100]
metrics = ["MSE", "SSIM", "PSNR"]
# metric labels with up and down arrows for best direction
metric_labels = {
    "MSE": r"MSE ($\downarrow$)",
    "SSIM": r"SSIM ($\uparrow$)",
    "PSNR": r"PSNR ($\uparrow$)",
}
metric_fmt = {
    "MSE": "{:.1e}",
    "SSIM": "{:.2f}",
    "PSNR": "{:.1f}",
}
metric_best = {
    "MSE": "min",
    "SSIM": "max",
    "PSNR": "max",
}


# ── validate dataframe columns ────────────────────────────────────────
required_cols = {"kernel", "components", "noise_std", *method_cols}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(f"Missing dataframe columns: {sorted(missing_cols)}")


# ── table dimensions ──────────────────────────────────────────────────
n_methods = len(method_cols)
n_noise = len(noise_levels)
n_label_cols = 2  # Kernel, Metric
n_data_cols = n_noise * n_methods
n_total_cols = n_label_cols + n_data_cols

# Explicit space between noise-level groups.
col_spec = "ll" + "".join(
    r"@{\hspace{0.8em}}ccc" for _ in noise_levels
)


# ── value formatting and best-result detection ────────────────────────
def is_valid_value(value):
    """Return True for finite numeric values."""
    try:
        return pd.notna(value) and np.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def format_value(value, metric):
    return metric_fmt[metric].format(float(value))


def format_cells(values, metric):
    """
    Format one method-comparison group for a single metric/noise level.

    Rules:
      - Unique best value: bold.
      - Tie for best at displayed precision: underline all tied best
        values instead of bolding them.
      - Missing/non-finite values: dash.
    """
    valid_methods = [
        m for m in method_cols if is_valid_value(values[m])
    ]

    if not valid_methods:
        return ["-"] * n_methods

    maximize = metric_best[metric] == "max"

    # Rank methods by their actual numerical values.
    ranked = sorted(
        valid_methods,
        key=lambda m: float(values[m]),
        reverse=maximize,
    )

    # Treat values that are identical at displayed precision as ties.
    displayed = {
        m: format_value(values[m], metric)
        for m in valid_methods
    }

    best_display = displayed[ranked[0]]
    best_methods = [
        m for m in valid_methods
        if displayed[m] == best_display
    ]

    styles = {}

    if len(best_methods) > 1:
        # Tied best values: underline all, rather than bolding them.
        for m in best_methods:
            styles[m] = "underline"

    else:
        # Unique best value: bold it.
        styles[best_methods[0]] = "bold"

        # Find the next-best distinct displayed value.
        remaining = [
            m for m in valid_methods
            if displayed[m] != best_display
        ]

        if remaining:
            for m in remaining:
                styles[m] = None

    cells = []

    for m in method_cols:
        if m not in valid_methods:
            cells.append("-")
            continue

        fmt = displayed[m]

        if styles.get(m) == "bold":
            fmt = rf"\textbf{{{fmt}}}"
        elif styles.get(m) == "underline":
            fmt = rf"\underline{{{fmt}}}"

        cells.append(fmt)

    return cells


# ── LaTeX table ───────────────────────────────────────────────────────
lines = [
    r"\begin{table*}",
    r"\centering",
    (
        r"\caption{\textbf{USPS digits data:} reconstruction metrics "
        r"across kernels, numbers of retained components, noise levels, "
        r"and methods (BWS corresponds to the method of \cite{bakir2004learning} and HR to that of \cite{honeine2011closed}). "
        r"Bold values indicate the best result for each metric and noise level, while underlined values indicate ties at the displayed precision.}"
    ),
    r"\label{tab:reconstruction}",
    r"\small",
    r"\setlength{\tabcolsep}{4.pt}",
    r"\renewcommand{\arraystretch}{1.08}",
    rf"\begin{{tabular}}{{{col_spec}}}",
    r"\toprule",
]

# ── Header row 1: noise levels ────────────────────────────────────────
noise_header = [r"\multicolumn{2}{c}{}"]

for noise in noise_levels:
    noise_header.append(
        rf"\multicolumn{{{n_methods}}}{{c}}{{$\sigma={noise:g}$}}"
    )

lines.append(" & ".join(noise_header) + r" \\")
cmidrules = []

for i in range(n_noise):
    start = n_label_cols + 1 + i * n_methods
    end = start + n_methods - 1
    cmidrules.append(rf"\cmidrule(lr){{{start}-{end}}}")

lines.append(" ".join(cmidrules))

# ── Header row 2: methods ─────────────────────────────────────────────
method_header = ["Kernel", "Metric"]

for _ in noise_levels:
    method_header.extend(method_names[m] for m in method_cols)

lines.append(" & ".join(method_header) + r" \\")
lines.append(r"\midrule")


# ── Data rows ─────────────────────────────────────────────────────────
for comp_idx, comp in enumerate(comp_levels):

    # Component-count section heading.
    lines.append(
        rf"\multicolumn{{{n_total_cols}}}{{l}}"
        rf"{{\textit{{$q = {comp}$}}}} \\"
    )
    lines.append(r"\addlinespace[2.5pt]")

    for k_idx, kernel in enumerate(kernel_order):

        for m_idx, metric in enumerate(metrics):

            # Put the kernel label across its three metric rows.
            if m_idx == 0:
                kernel_cell = (
                    rf"\multirow{{{len(metrics)}}}{{*}}"
                    rf"{{{kernel_names[kernel]}}}"
                )
            else:
                kernel_cell = ""

            cells = []

            for noise in noise_levels:

                # The metric is stored in df.index, as in the original code.
                mask = (
                    (df["kernel"] == kernel)
                    & (df["components"] == comp)
                    & (df.index == metric)
                    & np.isclose(
                        df["noise_std"].to_numpy(dtype=float),
                        noise,
                        rtol=0,
                        atol=1e-12,
                    )
                )

                noise_row = df.loc[mask]

                if len(noise_row) > 1:
                    raise ValueError(
                        "Duplicate results for "
                        f"kernel={kernel}, components={comp}, "
                        f"metric={metric}, noise_std={noise}."
                    )

                if noise_row.empty:
                    values = {m: np.nan for m in method_cols}
                else:
                    values = {
                        m: noise_row.iloc[0][m]
                        for m in method_cols
                    }

                cells.extend(format_cells(values, metric))

            row_cells = [kernel_cell, metric_labels[metric], *cells]
            lines.append(" & ".join(row_cells) + r" \\")

        # Add whitespace between kernel blocks.
        if k_idx < len(kernel_order) - 1:
            lines.append(r"\addlinespace[2.5pt]")

    # Separate component-count sections.
    if comp_idx < len(comp_levels) - 1:
        lines.append(r"\midrule")

lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table*}",
])

latex = "\n".join(lines) + "\n"

# Save the generated table.
output_path = Path(out_dir) / "usps_table_reconstruction_high.tex"
output_path.write_text(latex, encoding="utf-8")

print(f"Saved LaTeX table to {output_path}")

### Denoising output using the RBF kernel

In [ ]:
data, metrics = run_benchmark(kernel='rbf', n_components=100, noise_scale=0.25, gamma=gamma, noise_train=True,
                             alpha_bakir=1e-4, alpha_ours=1e-4, methods = 'all', n_alpha=100)

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
})

In [ ]:

rng = np.random.RandomState(seed=42)

# Select the same 25 test examples for every method
N_COLS, N_ROWS = 5, 5
idx = rng.choice(
    data["test"].shape[0],
    size=N_COLS * N_ROWS,
    replace=False,
)

def fmt_metrics(m):
    mse, ssim, psnr = m
    return f"MSE: {mse:.1e} | SSIM: {ssim:.2f}\nPSNR: {psnr:.1f} dB"


cols = [
    ("test",        "(a)", "Ground truth"),
    ("test_noisy",  "(b)", "Noisy test images"),
    ("ours",        "(c)", "Ours"),
    ("bakir",       "(d)", "Bakir et al."),
    ("mika",        "(e)", "Mika et al."),
    ("abrahamsen",  "(f)", "Abrahamsen et al."),
    ("kwok",        "(g)", "Kwok et al."),
    ("honeine",     "(h)", "Honeine et al."),
]

# Output directory
fig_dir = Path(out_dir) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Dimensions of each square image grid
PANEL_SIZE = 1.15  # inches

for key, _, _ in cols:
    dataset = data[key][idx]
    has_metrics = key != "test"

    # Leave room below the grid for two lines of metrics
    fig_height = PANEL_SIZE * (1.20 if has_metrics else 1.0)
    fig = plt.figure(figsize=(PANEL_SIZE, fig_height))

    if has_metrics:
        outer_gs = fig.add_gridspec(
            2, 1,
            height_ratios=[1, 0.20],
            left=0, right=1,
            bottom=0, top=1,
            hspace=0.02,
        )

        image_spec = outer_gs[0, 0]

        metric_ax = fig.add_subplot(outer_gs[1, 0])
        metric_ax.axis("off")
        metric_ax.text(
            0.5, 0.5,
            fmt_metrics(metrics[key]),
            transform=metric_ax.transAxes,
            fontsize=5.5,
            va="center",
            ha="center",
            style="italic",
        )
    else:
        image_spec = fig.add_gridspec(
            1, 1,
            left=0, right=1,
            bottom=0, top=1,
        )[0, 0]

    # Nested 10 x 10 image grid
    img_gs = gridspec.GridSpecFromSubplotSpec(
        N_ROWS, N_COLS,
        subplot_spec=image_spec,
        wspace=0.05,
        hspace=0.05,
    )

    for j, img in enumerate(dataset[:N_ROWS * N_COLS]):
        ax = fig.add_subplot(
            img_gs[j // N_COLS, j % N_COLS]
        )
        ax.imshow(
            img.reshape(16, 16),
            cmap="Greys",
            aspect="equal",
            interpolation="nearest",
        )
        ax.axis("off")

    # One independent PDF per method
    output_file = fig_dir / f"usps_denoising_{key}.pdf"

    fig.savefig(
        output_file,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.01,
    )
    plt.close(fig)

    print(f"Saved: {output_file}")